# GECS Task 1 — TF-IDF + LR Baseline (v9)

**Data:** cleaned_v9 | **Model:** TF-IDF word+char + Logistic Regression

Baseline for FLANG-BERT champion comparison.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import time
import warnings
import pickle
import json
import copy
from pathlib import Path
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score
from scipy.sparse import hstack
warnings.filterwarnings('ignore')

BASE_DIR    = Path('/content/drive/MyDrive/CAPSTONE')
CLEAN_DIR   = BASE_DIR / 'cleaned_v9'
FILE_T1     = CLEAN_DIR / 'task1_gecs_cleaned_v9.csv'
SPLITS_FILE = CLEAN_DIR / 'canonical_splits.npz'
ART_DIR     = BASE_DIR  / 'baseline_v9_artifacts'
ART_DIR.mkdir(parents=True, exist_ok=True)

# Find hierarchy file
for p in [BASE_DIR/'cleaned_v6'/'industries_Hierarchy.csv',
          BASE_DIR/'cleaned_v8'/'industries_Hierarchy.csv',
          BASE_DIR/'industries_Hierarchy.csv']:
    if p.exists():
        FILE_HIER = p
        break
else:
    FILE_HIER = None

print('Path check:')
print(('  OK' if FILE_T1.exists()     else '  NOT FOUND'), 'task1_gecs_cleaned_v9.csv')
print(('  OK' if SPLITS_FILE.exists() else '  NOT FOUND'), 'canonical_splits.npz')
print(('  OK' if FILE_HIER            else '  NOT FOUND'), 'industries_Hierarchy.csv')

In [ ]:
# Load data
t1      = pd.read_csv(FILE_T1, dtype={'MstarGlobal': str})
splits  = np.load(SPLITS_FILE)
t1_train_idx = splits['t1_train_idx']
t1_test_idx  = splits['t1_test_idx']

le = LabelEncoder()
le.fit(t1['MstarGlobal'])
t1['label'] = le.transform(t1['MstarGlobal'])
NUM_CLASSES  = len(le.classes_)

if FILE_HIER:
    hier = pd.read_csv(FILE_HIER, dtype={'industry_id': str})
    industry_to_sector = dict(zip(hier['industry_id'], hier['sector_name']))
else:
    industry_to_sector = {}

t1_top_k_str = t1['MstarGlobal'].value_counts().head(10).index.tolist()
t1_top_k_enc = le.transform(t1_top_k_str).tolist()

train_texts  = t1.iloc[t1_train_idx]['ModelInput'].fillna('').tolist()
test_texts   = t1.iloc[t1_test_idx]['ModelInput'].fillna('').tolist()
train_labels = t1.iloc[t1_train_idx]['label'].values
test_labels  = t1.iloc[t1_test_idx]['label'].values

print(f'Task 1 : {t1.shape}')
print(f'Train  : {len(t1_train_idx):,}  Test: {len(t1_test_idx):,}')
print(f'Classes: {NUM_CLASSES}')
print(f'Sample : {train_texts[0][:200]}')

In [ ]:
# TF-IDF Vectorization
print('[1/3] Fitting TF-IDF vectorizers...')
t0 = time.time()

word_vec = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=100000,
    sublinear_tf=True,
    min_df=2,
    analyzer='word',
)
char_vec = TfidfVectorizer(
    ngram_range=(3, 6),
    max_features=50000,
    sublinear_tf=True,
    min_df=3,
    analyzer='char_wb',
)

X_train_word = word_vec.fit_transform(train_texts)
X_train_char = char_vec.fit_transform(train_texts)
X_train      = hstack([X_train_word, X_train_char])

X_test_word  = word_vec.transform(test_texts)
X_test_char  = char_vec.transform(test_texts)
X_test       = hstack([X_test_word, X_test_char])

print(f'  word features : {X_train_word.shape[1]:,}')
print(f'  char features : {X_train_char.shape[1]:,}')
print(f'  total features: {X_train.shape[1]:,}')
print(f'  vectorize time: {time.time()-t0:.1f}s')

In [ ]:
# Logistic Regression with live checkpoint metrics
print('[2/3] Training LR with live metrics...')
print()
print(f'  {"Checkpoint":>12}  {"Status":>12}  {"Macro F1":>10}  {"Top10 F1":>10}  {"Accuracy":>10}  {"Time":>8}')
print(f'  {"-"*70}')

def evaluate_checkpoint(clf, X_test, test_labels, t1_top_k_enc):
    y_pred   = clf.predict(X_test)
    macro    = float(f1_score(test_labels, y_pred, average='macro',    zero_division=0))
    top10    = float(f1_score(test_labels, y_pred, labels=t1_top_k_enc, average='macro', zero_division=0))
    acc      = float(accuracy_score(test_labels, y_pred))
    return macro, top10, acc, y_pred

best_macro = 0.0
best_clf   = None
best_pred  = None

# Train at increasing iterations — liblinear is fast enough for checkpoints
for max_iter, label in [(10,'warmup'), (30,'early'), (75,'mid'),
                         (150,'late'), (300,'fine'), (500,'converge')]:
    t0 = time.time()
    clf = LogisticRegression(
        C=5,
        max_iter=max_iter,
        class_weight='balanced',
        solver='saga',
        n_jobs=-1,
        random_state=42,
    )
    clf.fit(X_train, train_labels)
    elapsed   = time.time() - t0
    converged = clf.n_iter_[0] < max_iter
    status    = 'CONVERGED' if converged else 'running'

    macro, top10, acc, y_pred = evaluate_checkpoint(clf, X_test, test_labels, t1_top_k_enc)
    flag = ' <- BEST' if macro > best_macro else ''

    print(f'  {label:>12}  {status:>12}  {macro:>10.4f}  {top10:>10.4f}  {acc:>10.4f}  {elapsed:>7.1f}s{flag}', flush=True)

    if macro > best_macro:
        best_macro = macro
        best_clf   = copy.deepcopy(clf)
        best_pred  = y_pred.copy()

    if converged:
        print(f'  Model converged at iteration {clf.n_iter_[0]} — stopping early.')
        break

clf    = best_clf
y_pred = best_pred
print(f'\nBest macro F1 : {best_macro:.4f}')

In [ ]:
# Full evaluation
print('[3/3] Full evaluation...')
y_true = test_labels

macro_f1 = float(f1_score(y_true, y_pred, average='macro',    zero_division=0))
micro_f1 = float(f1_score(y_true, y_pred, average='micro',    zero_division=0))
weighted = float(f1_score(y_true, y_pred, average='weighted', zero_division=0))
accuracy = float(accuracy_score(y_true, y_pred))
top10_f1 = float(f1_score(y_true, y_pred, labels=t1_top_k_enc, average='macro', zero_division=0))
per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
bottom50  = float(np.sort(per_class)[:50].mean())

print()
print('+-------------------------------------------------+')
print('|  Task 1 TF-IDF Baseline (v9 data)              |')
print('+-------------------------------------------------+')
print(f'|  macro F1       : {macro_f1:.4f}   {"ABOVE" if macro_f1>=0.75 else "below"} bar (0.75)      |')
print(f'|  top10_f1       : {top10_f1:.4f}   {"ABOVE" if top10_f1>=0.85 else "below"} bar (0.85)      |')
print(f'|  bottom-50 F1   : {bottom50:.4f}                          |')
print(f'|  micro F1       : {micro_f1:.4f}                          |')
print(f'|  weighted F1    : {weighted:.4f}                          |')
print(f'|  accuracy       : {accuracy:.4f}                          |')
print('+-------------------------------------------------+')

if industry_to_sector:
    y_true_str = le.inverse_transform(y_true)
    sectors    = pd.Series(y_true_str).map(industry_to_sector).values
    print('\n-- Per-Sector F1 --')
    for sec in sorted(pd.Series(sectors).dropna().unique()):
        mask = sectors == sec
        sc   = [ind for ind, s in industry_to_sector.items() if s == sec and ind in le.classes_]
        se   = le.transform(sc)
        sf1  = float(f1_score(y_true[mask], y_pred[mask], labels=se, average='macro', zero_division=0))
        flag = 'OK' if sf1 >= 0.75 else '  '
        print(f'  {flag} {sec:30s}  {sf1:.3f}  n={mask.sum():,}')

all_cls = sorted(np.unique(np.concatenate([y_true, y_pred])))
pc_f1   = f1_score(y_true, y_pred, labels=all_cls, average=None, zero_division=0)
pc_ser  = pd.Series(dict(zip(all_cls, pc_f1))).sort_values()

print('\n-- 20 Hardest Industries --')
for cls_enc, f1_val in pc_ser.head(20).items():
    cls_str = le.inverse_transform([cls_enc])[0]
    n_test  = (y_true == cls_enc).sum()
    flag    = ' sparse' if n_test < 10 else ''
    print(f'  {cls_str}  F1={f1_val:.3f}  n={n_test}{flag}')

print('\n-- 10 Easiest Industries --')
for cls_enc, f1_val in pc_ser.tail(10).items():
    cls_str = le.inverse_transform([cls_enc])[0]
    n_test  = (y_true == cls_enc).sum()
    print(f'  {cls_str}  F1={f1_val:.3f}  n={n_test}')

In [ ]:
# Save artifacts
with open(ART_DIR / 'tfidf_word_vec.pkl', 'wb') as f:
    pickle.dump(word_vec, f)
with open(ART_DIR / 'tfidf_char_vec.pkl', 'wb') as f:
    pickle.dump(char_vec, f)
with open(ART_DIR / 'logistic_regression.pkl', 'wb') as f:
    pickle.dump(clf, f)

pd.DataFrame({
    'CompanyId'  : t1.iloc[t1_test_idx]['CompanyId'].values,
    'SegmentName': t1.iloc[t1_test_idx]['SegmentName'].values,
    'y_true'     : le.inverse_transform(y_true),
    'y_pred'     : le.inverse_transform(y_pred),
    'correct'    : y_true == y_pred,
}).to_csv(ART_DIR / 'task1_tfidf_baseline_predictions.csv', index=False)

json.dump({
    'timestamp'  : datetime.now().isoformat(timespec='seconds'),
    'data'       : 'cleaned_v9_dual_signal',
    'model'      : 'TF-IDF word(1-3) + char(3-6) + LR(C=5, saga)',
    'n_train'    : int(len(t1_train_idx)),
    'n_test'     : int(len(t1_test_idx)),
    'macro_f1'   : round(macro_f1, 4),
    'top10_f1'   : round(top10_f1, 4),
    'bottom50_f1': round(bottom50, 4),
    'micro_f1'   : round(micro_f1, 4),
    'accuracy'   : round(accuracy, 4),
}, open(ART_DIR / 'baseline_summary.json', 'w'), indent=2)

print('All artifacts saved.')
print(f'Final macro F1: {macro_f1:.4f}')